<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Choose Two Clock Settings

**Choose two clock settings by solving one derivative equation for each setting.**

[Part 2](02_single_variable_analytical_optimization.ipynb) found the best constant correction. We keep its observations and squared-error score. Now we also allow the correction to change with elapsed time.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Add a second setting

Keep the recorded errors $-2,-1,-1,0$ min. They were observed at $t_i=0,1,2,3$ hours, for $i=1,\ldots,4$.

| Setting | Meaning | Unit |
|:---|:---|:---|
| $q$ | Correction at time zero | min |
| $r$ | Additional correction per hour | min/h |

Both settings may be any real number. If $d_i$ is the recorded error, the remaining error is

$$
y_i=d_i+q+rt_i.
$$

The two rows keep $q=1$ min fixed and show the effect of the rate setting. With $r=0$, each reading gets the same correction. With $r=-0.1$ min/h, the correction decreases by 0.1 min for each elapsed hour. The cards show $q+rt_i$, the correction added to each recorded error, before we calculate $y_i$.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-1_unconstrained_optimization_analytical_methods/assets/clock_offset_and_rate_controls.png" alt="Two clock controls set initial correction q and hourly rate r. With q equal to 1 minute, rate zero gives corrections 1, 1, 1, 1; rate minus 0.1 minute per hour gives 1, 0.9, 0.8, 0.7 minutes at elapsed times 0, 1, 2, 3 hours." width="480" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The offset sets the initial correction; the rate changes the correction over time. Both enter the same squared-error objective.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The definitions below collect the fixed observations, clock calculations, equation solver, and graph with its sliders. The error and time arrays have matching positions: the first error belongs to time zero.


In [ ]:
import numpy as np
import sys
import matplotlib

OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def objective_function(remaining_errors):
    return float(np.sum(np.asarray(remaining_errors) ** 2))


def solve_clock_correction(observed_errors=OBSERVED_ERRORS):
    return float(-np.mean(observed_errors))


def simulate_clock_with_rate(decision, observed_errors=OBSERVED_ERRORS,
                             observation_times=OBSERVATION_TIMES):
    correction, rate = np.asarray(decision, dtype=float)
    return np.asarray(observed_errors, dtype=float) + correction + rate * np.asarray(observation_times)


def evaluate_clock_with_rate(decision, observed_errors=OBSERVED_ERRORS,
                             observation_times=OBSERVATION_TIMES):
    response = simulate_clock_with_rate(decision, observed_errors, observation_times)
    times = np.asarray(observation_times, dtype=float)
    gradient = 2.0 * np.array([np.sum(response), np.dot(times, response)])
    return {
        "decision": np.asarray(decision, dtype=float),
        "response": response,
        "objective": objective_function(response),
        "gradient": gradient,
    }


def solve_clock_with_rate(observed_errors=OBSERVED_ERRORS,
                          observation_times=OBSERVATION_TIMES):
    errors = np.asarray(observed_errors, dtype=float)
    times = np.asarray(observation_times, dtype=float)
    # Coefficients in the two zero-gradient equations, after dividing by 2.
    coefficients = np.array([[len(errors), np.sum(times)],
                             [np.sum(times), np.dot(times, times)]], dtype=float)
    right_hand_side = -np.array([np.sum(errors), np.dot(times, errors)])
    return np.linalg.solve(coefficients, right_hand_side)


def _pyplot(*, interactive=False):
    # Match the course's Jupyter/VS Code and browser backend pattern.
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry
            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
GRAY = "#a6a6a6"


def show_clock_rate_analysis(decision=(1.0, 0.0), *, interactive=False, display=True):
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(1, 2, figsize=(12, 7.0 if interactive else 5.3))
    figure.suptitle("An offset and a rate correction reduce different parts of the clock error", fontsize=13)
    if interactive:
        figure.subplots_adjust(left=0.075, right=0.97, top=0.79,
                               bottom=0.34, wspace=0.30)
    status = figure.text(0.5, 0.89, "", ha="center", fontsize=10)
    optimum = evaluate_clock_with_rate(solve_clock_with_rate())
    axes[0].plot(OBSERVATION_TIMES, OBSERVED_ERRORS, "o--", color=BLUE,
                  alpha=0.6, label="Observed errors")
    axes[0].plot(OBSERVATION_TIMES, optimum["response"], "D-", color=TEAL,
                  label="Analytical minimum errors")
    error_line, = axes[0].plot([], [], "o-", color=ORANGE, label="Current corrected errors")
    axes[0].axhline(0.0, color=GRAY, linewidth=1)
    axes[0].set(xlabel="Elapsed time (h)", ylabel="Clock error (min)",
                xticks=OBSERVATION_TIMES, ylim=(-4, 5),
                title="A rate correction changes the error pattern")
    axes[0].legend(loc="upper left", fontsize=8)

    corrections = np.linspace(0.0, 3.0, 161)
    rates = np.linspace(-1.2, 0.6, 161)
    q_mesh, r_mesh = np.meshgrid(corrections, rates)
    score_mesh = sum((error + q_mesh + r_mesh * time) ** 2
                     for error, time in zip(OBSERVED_ERRORS, OBSERVATION_TIMES))
    contours = axes[1].contour(q_mesh, r_mesh, score_mesh,
                               levels=[0.3, 0.6, 1, 2, 4, 8, 16, 24], colors=BLUE, alpha=0.7)
    contour_labels = axes[1].clabel(contours, inline=True, fontsize=8, fmt="%g")
    axes[1].axhline(0.0, color=GRAY, linestyle="--", label="Offset-only choices: r = 0")
    axes[1].scatter(*optimum["decision"], color=TEAL, marker="D", s=90,
                    zorder=5, label="Analytical minimum")
    marker = axes[1].scatter([], [], color=ORANGE, s=70,
                              zorder=6, label="Current decision")
    axes[1].set(xlabel="Initial correction q (min)", ylabel="Rate correction r (min/h)",
                xlim=(0.0, 3.0), ylim=(-1.2, 0.6), title="Contours join equal scores (min²)")
    axes[1].legend(loc="upper right", fontsize=8)
    for axis in axes:
        axis.grid(alpha=0.25)

    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider
        q_axis = figure.add_axes([0.31, 0.20, 0.55, 0.03])
        r_axis = figure.add_axes([0.31, 0.12, 0.55, 0.03])
        sliders["correction"] = Slider(q_axis, "Correction q (min)", 0.0, 3.0,
                                       valinit=decision[0], valstep=0.05, color=BLUE)
        sliders["rate"] = Slider(r_axis, "Rate r (min/h)", -1.2, 0.6,
                                 valinit=decision[1], valstep=0.05, color=BLUE)
    else:
        figure.tight_layout(rect=(0, 0.08, 1, 0.83))

    # Hide contour labels that would be cut by the panel boundary.
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    panel = axes[1].get_window_extent(renderer)
    for label in contour_labels:
        box = label.get_window_extent(renderer)
        if (box.x0 < panel.x0 + 4 or box.x1 > panel.x1 - 4
                or box.y0 < panel.y0 + 4 or box.y1 > panel.y1 - 4):
            label.set_visible(False)

    figure.text(0.5, 0.035, "The displayed rectangle is not a constraint; both decisions range over the real numbers.",
                ha="center", fontsize=9)
    figure._analysis_state = {}

    def refresh(_=None):
        current_decision = [sliders["correction"].val, sliders["rate"].val] if sliders else decision
        record = evaluate_clock_with_rate(current_decision)
        error_line.set_data(OBSERVATION_TIMES, record["response"])
        marker.set_offsets([record["decision"]])
        q_value, r_value = record["decision"]
        q_slope, r_slope = np.where(np.abs(record["gradient"]) < 1e-10, 0.0, record["gradient"])
        status.set_text(f"q = {q_value:.2f} min | r = {r_value:.2f} min/h | "
                        f"score = {record['objective']:.2f} min² | "
                        f"gradient = ({q_slope:.2f}, {r_slope:.2f})")
        figure._analysis_state.update(current=record, optimum=optimum)
        figure.canvas.draw_idle()

    figure._analysis_sliders = sliders
    figure._analysis_refresh = refresh
    for slider in sliders.values():
        slider.on_changed(refresh)
    refresh()
    if display:
        plt.show()
    return figure


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Solve two small equations

Call the sum of squared remaining errors $\phi(q,r)$. Write the four terms and expand:

$$
\begin{aligned}
\phi(q,r)&=(q-2)^2+(q+r-1)^2\\
&\quad +(q+2r-1)^2+(q+3r)^2\\
&=4q^2+12qr+14r^2-8q-6r+6.
\end{aligned}
$$

For the derivative with respect to $q$, treat $r$ as a fixed number. For example, $12qr$ contributes $12r$ and $14r^2$ contributes zero.
For the derivative with respect to $r$, hold $q$ fixed instead:

$$
\frac{\partial\phi}{\partial q}=8q+12r-8,
\qquad
\frac{\partial\phi}{\partial r}=12q+28r-6.
$$

These two partial derivatives together are the **gradient**. At a differentiable unconstrained minimum, both must be zero.
Divide each equation by 2 to make the numbers smaller:

$$
\begin{aligned}
4q+6r&=4,\\
6q+14r&=3.
\end{aligned}
$$

The first equation gives $q=1-1.5r$. Substitute it into the second:

$$
6(1-1.5r)+14r=3
\quad\Longrightarrow\quad 5r=-3.
$$

Thus $r=-0.6$ min/h and $q=1.9$ min. This is our candidate; we will verify it below.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The equation solver uses `np.linalg.solve` to solve the two linear equations together. Compare the best offset-only setting $(q,r)=(1,0)$ with the new stationary candidate. Check both components of the gradient, not just the score.


In [ ]:
offset_only = evaluate_clock_with_rate([solve_clock_correction(), 0.0])
rate_candidate = evaluate_clock_with_rate(solve_clock_with_rate())
for name, record in (("Offset only", offset_only), ("Offset and rate", rate_candidate)):
    print(name)
    print("  decision [q (min), r (min/h)]:", np.round(record["decision"], 3))
    print("  remaining errors (min):", np.round(record["response"], 3))
    print("  score (min²):", round(record["objective"], 3))
    displayed_gradient = np.where(np.abs(record["gradient"]) < 1e-10, 0.0, record["gradient"])
    print("  gradient:", np.round(displayed_gradient, 10))


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Check that no other pair is better

The candidate $(q,r)=(1.9,-0.6)$ leaves errors $-0.1,0.3,-0.3,0.1$ min.
Their squares add to $0.01+0.09+0.09+0.01=0.2$ min².

As in the one-setting example, the score can be rewritten as a constant plus squares:

$$
\phi(q,r)=0.2+4(q+1.5r-1)^2+5(r+0.6)^2.
$$

The numerical coefficients use minutes and hours as above. Neither squared term can be negative, so the score is at least 0.2.
To reach 0.2, both squares must be zero. The last one gives $r=-0.6$; the other then gives $q=1.9$.
This proves the **unique global minimum**: every other pair has a larger score.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Inspect the two settings

The left panel compares remaining clock errors over the four observation times. The right panel is a **contour plot**: each curve joins decisions with the same score. The orange markers describe the current decision. The teal diamond marks the analytically verified minimum. The dashed line $r=0$ contains the original offset-only choices.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-1_unconstrained_optimization_analytical_methods/assets/clock_offset_and_rate.png" alt="Remaining clock errors for an offset-only decision and the analytical offset-and-rate solution, with score contours showing their positions in the two-decision plane." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The best offset-only setting has score 2 min². The verified two-setting solution has score 0.2 min². The tilted contours show that the best offset depends on the rate.

All markers, error values, and status values in the linked display use the same evaluation record. The contours are sampled only for drawing.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The blue sliders control the two settings shown in the graph. The displayed rectangle and slider steps are display settings, not constraints on the clock formulation.


In [ ]:
rate_explorer = show_clock_rate_analysis(decision=(1.0, 0.0), interactive=True)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The fitted settings minimize squared error on the four observations. They do not establish performance on future readings.
For more complicated smooth unconstrained objectives, the gradient equations may lack a convenient explicit solution.
The direct solution in this chapter relies on the stated quadratic objective.
